# Phase 2 — HumAID Dataset Exploration

## Goal

The goal of this phase is to understand the HumAID disaster-tweet dataset before
training or evaluating any model.

The exploration focuses on:

- Dataset structure and splits
- Class balance across humanitarian categories
- Event coverage
- Tweet length distribution
- Manual inspection of examples from each category
- Potential label ambiguity and modeling challenges

## 1. Loading the Dataset

HumAID is a dataset of disaster-related tweets categorized into humanitarian
information classes.

The dataset is loaded directly from the Hugging Face Hub using the
`QCRI/HumAID-all` dataset.

The available splits are:

- Train: 53,531 examples
- Dev: 7,793 examples
- Test: 15,160 examples

The dataset contains 76,484 tweets in total across these three splits.

In [ ]:
from datasets import load_dataset

train = load_dataset(
    "QCRI/HumAID-all",
    split="train",
    verification_mode="no_checks"
)

print(train)

## 2. Dataset Structure

The training split is converted into a pandas DataFrame to make exploration
and analysis easier.

The dataset contains two columns:

- `tweet_text` — the text of the disaster-related tweet
- `class_label` — the humanitarian category assigned to the tweet

In [ ]:
import pandas as pd

df = train.to_pandas()

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nClass counts:")
print(df["class_label"].value_counts())

print("\nClass percentages:")
print((df["class_label"].value_counts(normalize=True) * 100).round(2))

## 3. Class Balance

HumAID contains multiple humanitarian categories, and the classes are not
uniformly distributed.

Examining the class distribution is important because a model can achieve
good overall performance by performing well on common classes while performing
poorly on rare classes.

Therefore, class imbalance will be particularly important when interpreting
metrics such as macro-F1 and weighted-F1 during later model evaluation.

In [ ]:
print("Columns:", df.columns.tolist())

if "event" in df.columns:
    print("\nNumber of unique events:", df["event"].nunique())
    print("\nTweets per event:")
    print(df["event"].value_counts())
else:
    print("\nNo column named 'event' found.")

### Class Balance Findings

The training set contains 53,531 tweets across 10 observed humanitarian
categories.

The most common category is `rescue_volunteering_or_donation_effort`, with
14,891 tweets (27.82%).

The least common category is `missing_or_found_people`, with only 250 tweets
(0.47%).

This shows a substantial class imbalance. The largest class contains roughly
60 times more examples than the smallest class.

This imbalance should be considered when evaluating models. In particular,
macro-F1 will be useful because it gives equal importance to each class,
including rare categories.

## 4. Event Coverage

The dataset representation loaded in this environment contains only the
following columns:

- `tweet_text`
- `class_label`

No `event` or disaster-event identifier column is available.

Therefore, event-level distribution and event-specific analysis cannot be
performed directly using this dataset representation.

This is an important limitation to record because event-level distribution
shift cannot be analyzed from the current columns alone.

## 5. Tweet Length Distribution

Tweet length is examined to understand the characteristics of the input text.

The number of words in each tweet is calculated and summarized using
descriptive statistics.

The distribution can help identify whether the dataset mainly contains short
messages or whether there is substantial variation in tweet length.

This is also relevant for later model experiments because tokenized sequence
length affects memory usage and computational cost.

In [ ]:
import matplotlib.pyplot as plt

df["word_length"] = df["tweet_text"].str.split().str.len()

plt.hist(df["word_length"], bins=30)
plt.xlabel("Word count")
plt.ylabel("Number of tweets")
plt.title("HumAID tweet length distribution")
plt.show()

print(df["word_length"].describe())

In [ ]:
for label in df["class_label"].unique():
    print(f"\n{'=' * 80}")
    print(f"=== {label} ===")
    print(f"{'=' * 80}")

    examples = df[df["class_label"] == label].sample(
        min(5, len(df[df["class_label"] == label])),
        random_state=42
    )["tweet_text"].tolist()

    for i, tweet in enumerate(examples, 1):
        print(f"{i}. {tweet}")

## Step 7 — Dataset Exploration Summary

### Class balance

The HumAID training split contains **53,531 tweets** across 10 observed humanitarian categories.

The largest category is `rescue_volunteering_or_donation_effort` with **14,891 tweets (27.82%)**, followed by `other_relevant_information` with **8,501 tweets (15.88%)** and `sympathy_and_support` with **6,250 tweets (11.68%)**.

The dataset is substantially imbalanced. The smallest category is `missing_or_found_people`, with only **250 tweets (0.47%)**, while the largest category contains more than 14,000 tweets. This imbalance means that accuracy and weighted F1 may hide poor performance on rare categories, so macro F1 will be particularly important during model evaluation.

### Event coverage

The dataset columns are `tweet_text` and `class_label`. No `event` column is available in the loaded dataset, so event-level distribution could not be directly analyzed from these columns.

### Tweet length

Tweets are generally short. The average tweet contains **22.11 words**, with a median of **19 words**. The middle 50% of tweets contain between **14 and 28 words**. The shortest observed tweet contains 3 words and the longest contains 96 words.

### Manual label review

Manual inspection showed that many examples have clear category labels, such as donation/help requests, sympathy messages, and reports of deaths or injuries. However, some tweets contain multiple types of humanitarian information, which can make the primary category ambiguous. The dataset also contains noisy real-world Twitter text such as hashtags, mentions, retweets, news headlines, and emojis, which may make classification more challenging.